# Prompt Chaining
Prompt chaining represents one of the most fundamental agentic workflow patterns, where complex tasks are decomposed into manageable sequential steps. Each step builds upon the previous one, creating a structured approach that significantly improves accuracy by simplifying individual subtasks before progressing to the next phase.

This pattern excels in scenarios where step-by-step reasoning enhances overall output quality. Rather than overwhelming the language model with a complex, multi-faceted prompt, prompt chaining breaks down the cognitive load into digestible components, allowing for more focused and accurate processing at each stage.

![image.png](attachment:image.png)

| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), "Prompt chaining" section |

#### How Prompt Chaining Works with LangGraph
1. Define the Task: Start by breaking down the problem into smaller sub-tasks. For example, if you want to generate a detailed report, you might split it into steps like "gather data," "analyze data," and "write summary."

2. Create Nodes: Each sub-task becomes a node in the LangGraph structure. A node could be a prompt that instructs the model to perform a specific action, such as "List key facts about X" or "Summarize the following text."

3. Establish Edges: Edges define the sequence and dependencies between nodes. For instance, the output of the "gather data" node flows into the "analyze data" node, ensuring the model has the necessary context to proceed.

4. Execute the Graph: LangGraph processes the nodes in order, passing information along the edges. The model generates responses step-by-step, refining the output as it progresses through the chain.

5. Iterate if Needed: LangGraph supports conditional logic and loops, so you can revisit earlier nodes or adjust the flow based on intermediate results.




### Step 1: Setting Up the Environment


In [ ]:
from dotenv import load_dotenv
import operator
from typing import Annotated

from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
# from langchain_google_genai import ChatGoogleGenerativeAI

# load_dotenv()

# # Initialize the language model
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
# ============================================================================
# SETUP: Import LLM Helper Functions
# ============================================================================
# We use helper functions to create LLM instances with proper configuration
# These functions handle API key loading and model configuration

import os
import sys

# Add parent directory to path for importing helpers
sys.path.append(os.path.abspath(".."))

# Import our LLM factory functions
# - get_groq_llm(): Creates a Groq-hosted LLM (fast inference)
# - get_openai_llm(): Creates an OpenAI GPT model
# - get_databricks_gateway_llm(): AI Gateway (system.ai.* models)
# - get_databricks_llm(): Model Serving endpoints only (not system.ai.*)
from helpers.utils import get_groq_llm, get_openai_llm, get_databricks_gateway_llm

print("LLM helpers imported successfully!")

# ============================================================================
# CREATE THE LLM AND CHATBOT GRAPH
# ============================================================================

# -----------------------------------------------------------------------------
# Step 1: Initialize the LLM
# Use the AI Gateway helper for system.ai.* catalog models.
# ChatDatabricks (get_databricks_llm) looks up a serving endpoint and 404s.
# -----------------------------------------------------------------------------
llm = get_databricks_gateway_llm("system.ai.meta-llama-3-3-70b-instruct")
# Alternative: llm = get_openai_llm()  # OpenAI's GPT models

if hasattr(llm, 'model_name'):
    print(f"LLM initialized: {llm.model_name}")
elif hasattr(llm, 'model'):
    print(f"LLM initialized: {llm.model} (Databricks)")
else:
    print("LLM initialized: Groq LLM")

In [ ]:
llm.invoke("hello")


### Step 2: Defining the Workflow State

In [ ]:
class EmailState(TypedDict):
    """State to track email creation progress"""
    topic: str
    key_points: str
    draft_email: str
    final_email: str
    # operator.add so each extract() return of 1 accumulates (1, 2, 3…)
    # Last-value overwrite was getting dropped on the regenerate loop.
    retry_count: Annotated[int, operator.add]


MAX_VALIDATION_RETRIES = 3

### Step 3: Creating Processing Functions

In [ ]:
def extract_key_points(state: EmailState):
    """Step 1: Extract key points from the topic"""
    attempt = state.get("retry_count", 0) + 1
    prompt = f"List 3 key topics about: {state['topic']}"
    response = llm.invoke(prompt)
    print(f"✅ Key points extracted (attempt {attempt}/{MAX_VALIDATION_RETRIES})")
    # Return +1; EmailState uses operator.add so the graph sums visits.
    return {"key_points": response.content, "retry_count": 1}

def validate_key_points(state: EmailState):
    """Quality check: Ensure key points are actionable and specific"""
    key_points = state["key_points"]
    attempt = state.get("retry_count", 0)

    # Check for actionable words and specific content
    actionable_words = ['request', 'need', 'require', 'propose', 'suggest', 'recommend', 'deadline', 'schedule', 'meeting', 'discuss', 'review', 'approve', 'extension', 'support', 'assistance', 'feedback', 'update', 'status', 'progress', 'complete', 'deliver',"deadline"]

    # Convert to lowercase for checking
    points_lower = key_points.lower()

    # Count actionable words found
    actionable_count = sum(1 for word in actionable_words if word in points_lower)

    # Also check minimum length (should be substantial)
    word_count = len(key_points.split())

    if actionable_count >= 2 and word_count >= 15:
        print(f"✅ Key points validation: PASSED (Found {actionable_count} actionable words, {word_count} total words)")
        return "write_draft"

    if attempt >= MAX_VALIDATION_RETRIES:
        print(
            f"❌ Key points validation: FAILED (Only {actionable_count} actionable words, "
            f"{word_count} total words) — stopping after {attempt} attempts"
        )
        return "fail"

    print(
        f"❌ Key points validation: FAILED (Only {actionable_count} actionable words, "
        f"{word_count} total words) - regenerating... ({attempt}/{MAX_VALIDATION_RETRIES})"
    )
    return "regenerate"

def fail_validation(state: EmailState):
    """Stop the chain when retries are exhausted."""
    msg = (
        f"Stopped: could not extract actionable key points after "
        f"{state.get('retry_count', 0)} attempts. "
        f"Topic may be too abstract: '{state['topic']}'"
    )
    print(f"🛑 {msg}")
    return {"final_email": msg}

def write_draft(state: EmailState):
    """Step 2: Write email draft using key points"""
    prompt = f"Write a professional email draft covering these points: {state['key_points']}"
    response = llm.invoke(prompt)
    print(f"✅ Draft written")
    return {"draft_email": response.content}

def polish_email(state: EmailState):
    """Step 3: Polish and add proper formatting"""
    prompt = f"Polish this email and add proper greeting/closing: {state['draft_email']}"
    response = llm.invoke(prompt)
    print(f"✅ Email polished")
    return {"final_email": response.content}

### Step 4: Building the Sequential Workflow

In [ ]:
# Create the workflow graph
workflow = StateGraph(EmailState)

# Add processing nodes
workflow.add_node("extract_points", extract_key_points)
workflow.add_node("write_draft", write_draft)
workflow.add_node("polish_final", polish_email)
workflow.add_node("fail_validation", fail_validation)

# Connect nodes with conditional logic
workflow.add_edge(START, "extract_points")
workflow.add_conditional_edges(
    "extract_points",
    validate_key_points,
    {
        "write_draft": "write_draft",
        "regenerate": "extract_points",
        "fail": "fail_validation",
    },
)
workflow.add_edge("write_draft", "polish_final")
workflow.add_edge("polish_final", END)
workflow.add_edge("fail_validation", END)

# Compile the workflow
compiled_workflow = workflow.compile()

In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        compiled_workflow.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

### Step 5: Running the Workflow with Test Cases

In [ ]:
# Execute the email creation process
def create_email(topic: str):
    """Run the complete email creation workflow"""
    print(f"📧 Creating email about: '{topic}'")
    print("-" * 40)
    
    result = compiled_workflow.invoke(
        {"topic": topic, "retry_count": 0},
        {"recursion_limit": MAX_VALIDATION_RETRIES * 4},
    )
    
    print("\n🎉 Email creation completed!")
    return result["final_email"]


In [ ]:
# Test Case 1: Should pass validation (specific, detailed topic)
print("🧪 TEST CASE 1: Detailed Topic (Should Pass)")
email1 = create_email("Request for project deadline extension due to technical challenges")
print(f"\n📄 FINAL EMAIL:\n{email1}")

In [ ]:
# Test Case 2: Should fail validation (abstract/philosophical topic)
# Guardrail: regenerate at most MAX_VALIDATION_RETRIES times, then stop.
print("🧪 TEST CASE 2: Abstract Topic (Should Fail & Stop After Max Retries)")
email2 = create_email("The meaning of life and happiness")
print(f"\n📄 FINAL EMAIL:\n{email2}")